# Day 11: Map-Reduce with `Send`

Yesterday we ran three nodes in parallel: pros, cons and risks. We knew there would be exactly three, so we drew three edges.

But a lot of real problems don't work like that:

- A user uploads some files and wants each one summarized. Is it 2 files or 20?
- An LLM plans a report and decides it needs 4 sections. Or 6.

You can't draw edges for a number you don't know yet. That's the problem `Send` solves.

Picture a teacher with a pile of answer sheets. She doesn't know in advance how many students will turn up to the exam. When the sheets come in, she hands one to each available helper to grade, all at the same time. When every helper is done, she collects the marks and calculates the class average.

That's **map-reduce**:

- **Map**: split the work into pieces and process each piece in parallel
- **Reduce**: collect all the results and combine them

In LangGraph, a router returns a list of `Send("node_name", input)` objects, one per piece of work. LangGraph runs that node once per `Send`, all in parallel, each with its own input.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. `Send` without any LLM

Let's learn the mechanics with something simple: square every number in a list, one worker per number.

```
              +--> square(1) --+
  START ------+--> square(2) --+--> add_up --> END
              +--> square(N) --+
```

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [ ]:
class MainState(TypedDict):
    numbers: list[int]
    squares: Annotated[list[int], operator.add]    # collects the results from every worker
    total: int

Each worker only needs one number, so we give the workers their own small input type. The worker does **not** see the whole main state, only what we hand it.

In [ ]:
class WorkerInput(TypedDict):
    n: int

Now the router. Instead of returning one node name, it returns a list of `Send` objects: one per number.

In [ ]:
def hand_out_work(state: MainState) -> list[Send]:
    return [Send("square", {"n": number}) for number in state["numbers"]]

In [ ]:
def square(work: WorkerInput) -> dict:
    print(f"  worker squaring {work['n']}")
    # The result goes back into the MAIN state, through the squares reducer
    return {"squares": [work["n"] ** 2]}

In [ ]:
def add_up(state: MainState) -> dict:
    return {"total": sum(state["squares"])}

In [ ]:
builder = StateGraph(MainState)

In [ ]:
builder.add_node("square", square)
builder.add_node("add_up", add_up)

In [ ]:
builder.add_conditional_edges(START, hand_out_work, ["square"])
builder.add_edge("square", "add_up")
builder.add_edge("add_up", END)

In [ ]:
squares_app = builder.compile()
show_graph(squares_app)

### Predict before you run

We pass `[1, 2, 3, 4, 5]`. How many times does `square` run? How many times does `add_up` run? And what's the total?

<details>
<summary>Click to see the answer</summary>

`square` runs 5 times (one per number, all in the same step). `add_up` runs only **once**, after all 5 workers have finished. The total is 1 + 4 + 9 + 16 + 25 = 55.

The order of the "worker squaring" lines may vary, because they run in parallel.
</details>

In [ ]:
result = squares_app.invoke({"numbers": [1, 2, 3, 4, 5]})
print(result)

### Your turn

Try a list with 10 numbers, or an empty list. What happens with an empty list? (Think before you run: if there are no `Send`s, does `add_up` run at all?)

In [ ]:
my_numbers = [10, 20, 30]      # try [] or list(range(10))
print(squares_app.invoke({"numbers": my_numbers}))

Three things to remember from this example:

- The worker gets **only** what you put inside `Send(...)`, not the whole state.
- The worker writes back to the **main** state, so the key it writes to needs a reducer (`operator.add`) to collect everything.
- The next node (`add_up`) waits until **all** workers are finished.

## 2. Map-reduce with an LLM: reviewing many reviews

Now something realistic. We have a bunch of product reviews. We analyse each one in parallel with structured output (map), then write one overall summary for the product manager (reduce).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
class ReviewResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    issue: str = Field(description="the main complaint or praise, in at most 6 words")

In [ ]:
analyzer = llm.with_structured_output(ReviewResult)

In [ ]:
class ReviewsState(TypedDict):
    reviews: list[str]
    results: Annotated[list[dict], operator.add]
    summary: str

In [ ]:
def send_each_review(state: ReviewsState) -> list[Send]:
    return [Send("analyze", {"review": review}) for review in state["reviews"]]

In [ ]:
def analyze(work: dict) -> dict:
    result = analyzer.invoke(f"Analyze this product review: {work['review']}")
    # model_dump() turns the Pydantic object into a normal dictionary
    return {"results": [{"review": work["review"], **result.model_dump()}]}

In [ ]:
def summarize(state: ReviewsState) -> dict:
    lines = "\n".join(f"- {r['sentiment']}: {r['issue']}" for r in state["results"])
    prompt = f"Here are review analyses:\n{lines}\n\nWrite a 3-sentence summary for the product manager."
    return {"summary": llm.invoke(prompt).content}

In [ ]:
builder = StateGraph(ReviewsState)

In [ ]:
builder.add_node("analyze", analyze)
builder.add_node("summarize", summarize)

In [ ]:
builder.add_conditional_edges(START, send_each_review, ["analyze"])
builder.add_edge("analyze", "summarize")
builder.add_edge("summarize", END)

In [ ]:
reviews_app = builder.compile()

In [ ]:
reviews = [
    "Battery lasts two full days, love it!",
    "Screen cracked after one week. Very poor quality.",
    "Camera is decent for the price.",
    "Delivery was late and the box was damaged.",
    "Super fast charging, great value for money.",
]

result = reviews_app.invoke({"reviews": reviews})

for r in result["results"]:
    print(f"{r['sentiment']:8} | {r['issue']}")

print()
print(result["summary"])

## 3. Orchestrator and workers: a report writer

This is the pattern I find most useful in practice. It's how a newspaper editor works:

1. The **orchestrator** (the editor) looks at the topic and plans the sections.
2. **Workers** (the reporters) each write one section, all at the same time.
3. A **synthesizer** (the sub-editor) puts the sections together into the final article.

The difference from section 2 is that the list of tasks isn't given to us. The LLM **plans** it.

```
  START -> orchestrator --(Send x N)--> write_section -> synthesizer -> END
```

In [ ]:
class Section(BaseModel):
    title: str = Field(description="the section heading")
    brief: str = Field(description="one sentence on what this section should cover")

In [ ]:
class Plan(BaseModel):
    sections: list[Section] = Field(description="3 or 4 sections for a short report")

In [ ]:
planner = llm.with_structured_output(Plan)

In [ ]:
class ReportState(TypedDict):
    topic: str
    sections: list[Section]
    written: Annotated[list[str], operator.add]
    report: str

In [ ]:
def orchestrator(state: ReportState) -> dict:
    plan = planner.invoke(f"Plan a short report on: {state['topic']}")
    print("Planned sections:", [section.title for section in plan.sections])
    return {"sections": plan.sections}

In [ ]:
def assign_writers(state: ReportState) -> list[Send]:
    return [
        Send("write_section", {"section": section, "topic": state["topic"]})
        for section in state["sections"]
    ]

In [ ]:
def write_section(work: dict) -> dict:
    section = work["section"]
    prompt = (
        f"Report topic: {work['topic']}\n"
        f"Write the section '{section.title}' ({section.brief}). "
        "At most 70 words. Don't include the heading."
    )
    body = llm.invoke(prompt).content
    return {"written": [f"## {section.title}\n{body}"]}

In [ ]:
def synthesizer(state: ReportState) -> dict:
    # The workers can finish in any order, so put the sections back in the planned order
    planned_order = [section.title for section in state["sections"]]

    def position(text: str) -> int:
        title = text.split("\n")[0].replace("## ", "")
        return planned_order.index(title) if title in planned_order else len(planned_order)

    ordered = sorted(state["written"], key=position)
    return {"report": f"# {state['topic']}\n\n" + "\n\n".join(ordered)}

In [ ]:
builder = StateGraph(ReportState)

In [ ]:
builder.add_node("orchestrator", orchestrator)
builder.add_node("write_section", write_section)
builder.add_node("synthesizer", synthesizer)

In [ ]:
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", assign_writers, ["write_section"])
builder.add_edge("write_section", "synthesizer")
builder.add_edge("synthesizer", END)

In [ ]:
report_app = builder.compile()
show_graph(report_app)

In [ ]:
final = report_app.invoke({"topic": "How UPI changed payments in India"})
print(final["report"])

A small extra: a node can also send work directly, by returning `Command(goto=[Send(...), Send(...)])`. You'll see that in the Day 16 capstone.

## 4. Common mistakes

**Mistake 1: No reducer on the key the workers write to.**
All the workers write to the same key in the same step. Without a reducer, LangGraph can't combine them (same problem as Day 2).

In [ ]:
from langgraph.errors import InvalidUpdateError

In [ ]:
class NoReducerState(TypedDict):
    numbers: list[int]
    squares: list[int]          # forgot Annotated[..., operator.add]

In [ ]:
def hand_out_numbers(state: NoReducerState) -> list[Send]:
    return [Send("square", {"n": number}) for number in state["numbers"]]

In [ ]:
builder = StateGraph(NoReducerState)

In [ ]:
builder.add_node("square", square)

In [ ]:
builder.add_conditional_edges(START, hand_out_numbers, ["square"])
builder.add_edge("square", END)

In [ ]:
broken_app = builder.compile()

In [ ]:
try:
    broken_app.invoke({"numbers": [1, 2, 3]})
except InvalidUpdateError as error:
    print("InvalidUpdateError: the workers collided on the 'squares' key")

**Fix:** `squares: Annotated[list[int], operator.add]`.

**Mistake 2: Expecting the worker to see the whole state.**
Inside `square`, `work["numbers"]` doesn't exist. The worker only gets the dictionary you put in `Send`. If it needs something else, like the topic in the report writer, put it in the `Send` too.

**Mistake 3: Forgetting the list of destinations.**
In `add_conditional_edges(START, hand_out_work, ["square"])`, that last list tells LangGraph where the `Send`s might go. Without it the graph still runs, but the diagram can't draw the connection, which makes your graph harder to understand.

**Mistake 4: Assuming results come back in order.**
Parallel workers finish whenever they finish. If order matters, as with report sections, sort the results yourself, like the synthesizer does.

## 5. Practice

**Exercise 1 (no LLM).** Given a list of words, use `Send` to count the letters of each word in parallel, then a final node picks the longest word.

**Exercise 2.** Given three cities, use `Send` to ask the LLM for one must-try street food in each city, and collect them in a list.

In [ ]:
# Solution 1
class WordsState(TypedDict):
    words: list[str]
    lengths: Annotated[list[tuple], operator.add]
    longest: str

In [ ]:
def send_each_word(state: WordsState) -> list[Send]:
    return [Send("count_letters", {"word": word}) for word in state["words"]]

In [ ]:
def count_letters(work: dict) -> dict:
    return {"lengths": [(work["word"], len(work["word"]))]}

In [ ]:
def pick_longest(state: WordsState) -> dict:
    longest_word = ""
    for word, length in state["lengths"]:
        if length > len(longest_word):
            longest_word = word
    return {"longest": longest_word}

In [ ]:
builder = StateGraph(WordsState)

In [ ]:
builder.add_node("count_letters", count_letters)
builder.add_node("pick_longest", pick_longest)

In [ ]:
builder.add_conditional_edges(START, send_each_word, ["count_letters"])
builder.add_edge("count_letters", "pick_longest")
builder.add_edge("pick_longest", END)

In [ ]:
words_app = builder.compile()

In [ ]:
result = words_app.invoke({"words": ["graph", "langgraph", "node", "edge"]})
print(result["longest"])
assert result["longest"] == "langgraph"

In [ ]:
# Solution 2
class FoodState(TypedDict):
    cities: list[str]
    foods: Annotated[list[str], operator.add]

In [ ]:
def send_each_city(state: FoodState) -> list[Send]:
    return [Send("find_food", {"city": city}) for city in state["cities"]]

In [ ]:
def find_food(work: dict) -> dict:
    food = llm.invoke(f"Name ONE must-try street food in {work['city']}. Reply in at most 5 words.").content
    return {"foods": [f"{work['city']}: {food}"]}

In [ ]:
builder = StateGraph(FoodState)

In [ ]:
builder.add_node("find_food", find_food)

In [ ]:
builder.add_conditional_edges(START, send_each_city, ["find_food"])
builder.add_edge("find_food", END)

In [ ]:
food_app = builder.compile()

In [ ]:
result = food_app.invoke({"cities": ["Mumbai", "Kolkata", "Lucknow"]})
for line in result["foods"]:
    print(line)

## Wrapping up

When you don't know how many parallel tasks you'll need until the program is running, have a router return a list of `Send("node", input)` objects. Each one starts its own run of that node with its own input. The workers write back to the main state through a **reducer**, and the next node waits for all of them.

That gives you **map-reduce** (split, process in parallel, combine) and the **orchestrator-workers** pattern (an LLM plans the tasks, workers do them, a synthesizer joins them up).

Tomorrow we look at bigger systems: building graphs out of smaller graphs (**subgraphs**), and getting several specialised agents to work together as a team.